In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch

from base import CUDA, DTYPE
from FreeFermion.linalg import (randcov, williamson_decomposition, williamson_modes,
                                williamson_modular, pfaffian,
                                _mode_coefficients, _vacuum_contraction)
from FreeFermion.cuda import fermion

## `FreeFermion.linalg`: sampling, Williamson's theorem, Wick/Pfaffian contraction

| function | what it returns |
|---|---|
| `randcov(dim)` | random pure covariance `Gamma = q J q^T` of `dim` Majorana modes |
| `williamson_decomposition(Gamma)` | `(R, D)` with `Gamma = R^T D R`, `D` block diagonal in `lambda_k [[0, -1], [1, 0]]` |
| `williamson_modes(Gamma)` | `(occupations, coeff)`: normal modes `d_k`, `p_k = (1 - lambda_k) / 2` |
| `williamson_modular(Gamma)` | `(W, lambdas)` with the modular matrix `W = -2i artanh(i Gamma)` |
| `pfaffian`, `_mode_coefficients`, `_vacuum_contraction` | the Wick contraction chain, already covered by the CUDA kernels in `FreeFermion/cuda/fermion.cu` |


### 1. Sampling and the Williamson decomposition


In [2]:
A = randcov(6)                       # pure covariance of 6 Majorana modes
R, D = williamson_decomposition(A)

print('|Gamma - R^T D R| =', float((A - R.T @ D @ R).abs().max()))
print('|R^T R - I|       =', float((R.T @ R - torch.eye(6, dtype=R.dtype, device=CUDA)).abs().max()))
print('lambdas           =', [round(float(x), 12) for x in -D[0::2, 1::2].diagonal()])
print('D =')
print(D) # notice the block diag structure


|Gamma - R^T D R| = 8.881784197001252e-16
|R^T R - I|       = 8.881784197001252e-16
lambdas           = [1.0, 1.0, 1.0]
D =
tensor([[ 0.0000, -1.0000,  0.0000, -0.0000,  0.0000, -0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.0000,  0.0000, -1.0000,  0.0000, -0.0000],
        [ 0.0000,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -1.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000]],
       device='cuda:0', dtype=torch.float64)


### 2. Normal modes and the modular matrix


In [3]:
# a pure covariance: every lambda_k = 1, hence every mode is empty
occupations, coeff = williamson_modes(A)
W, lambdas = williamson_modular(A)

print('occupations p_k          =', [round(float(x), 12) for x in occupations])
print('|2 c c^dag - I|           =', float((2.0 * (coeff @ coeff.conj().T)
                                          - torch.eye(3, dtype=coeff.dtype, device=CUDA)).abs().max()))
print('|W + W^T| (antisymmetry)  =', float((W + W.T).abs().max()))

# a mixed state instead: a 3-mode patch of the Kitaev ground state
from CGO.GaussianCGO import free_fermion_hamiltonian, ground_state_covariance

patch = ground_state_covariance(free_fermion_hamiltonian(60, 1.0, 0.5, 2.5, CUDA))[58:64, 58:64]
patch_occupations, patch_coeff = williamson_modes(patch)
patch_W, patch_lambdas = williamson_modular(patch)
print('patch lambdas            =', [round(float(x), 9) for x in patch_lambdas])
print('patch occupations        =', [round(float(x), 9) for x in patch_occupations])
print('patch |W + W^T|          =', float((patch_W + patch_W.T).abs().max()))


occupations p_k          = [0.0, -0.0, -0.0]
|2 c c^dag - I|           = 6.661540996356336e-16
|W + W^T| (antisymmetry)  = 4.626774047084798e-15
patch lambdas            = [0.931703597, 0.980561637, 0.999993263]
patch occupations        = [0.034148202, 0.009719182, 3.369e-06]
patch |W + W^T|          = 4.440892098500626e-16


### 3. The Wick/Pfaffian chain (CUDA kernels)


In [4]:
def by_pairings(matrix: torch.Tensor) -> torch.Tensor:
    '''
    brute-force Pfaffian by enumerating the pairings, only for the 4x4 cross-check.
    '''
    size = matrix.shape[0]
    if size == 0:
        return torch.ones((), dtype=matrix.dtype, device=matrix.device)
    total = torch.zeros((), dtype=matrix.dtype, device=matrix.device)
    for j in range(1, size):
        keep = [k for k in range(1, size) if k != j]
        total = total + (-1) ** (j + 1) * matrix[0, j] * by_pairings(matrix[keep][:, keep])
    return total

torch.manual_seed(0)
antisymmetric = torch.randn(4, 4, dtype=DTYPE, device=CUDA)
antisymmetric = antisymmetric - antisymmetric.T
print('kernel pfaffian =', complex(pfaffian(antisymmetric)))
print('by pairings     =', complex(by_pairings(antisymmetric)))


kernel pfaffian = (-2.0021356531157175-2.7922848744741593j)
by pairings     = (-2.0021356531157166-2.7922848744741593j)


In [5]:
# one empty mode: <0|gamma_0 gamma_1|0> = 1j
single = torch.tensor([[0.0, -1.0], [1.0, 0.0]], dtype=torch.float64, device=CUDA)
occupations, coeff = williamson_modes(single)
gamma_0 = _mode_coefficients(coeff, 1, ('gamma', 0))
gamma_1 = _mode_coefficients(coeff, 1, ('gamma', 1))

print('gamma_0 in the mode basis =', gamma_0.tolist())
print('<0|gamma_0 gamma_1|0>     =',
      complex(pfaffian(_vacuum_contraction([gamma_0, gamma_1], 1))), '(expected 1j)')


gamma_0 in the mode basis = [(-1-0j), (-1+0j)]
<0|gamma_0 gamma_1|0>     = 1j (expected 1j)


### 4. Batched calls, which is what the kernels are for


In [6]:
# 8 identical operator lists [gamma_0, gamma_1], one call for the whole batch
codes = torch.tensor([[[2, 0], [2, 1]]] * 8, dtype=torch.long, device=CUDA)   # kind 2 = gamma_mu
vectors = fermion.operator_vectors(coeff, codes)      # (8, 2, 2)
matrices = fermion.vacuum_contraction(vectors, 1)     # (8, 2, 2)
values = fermion.pfaffian(matrices)                   # (8,)
print('shapes:', tuple(vectors.shape), tuple(matrices.shape), tuple(values.shape))
print('values:', values.tolist())


shapes: (8, 2, 2) (8, 2, 2) (8,)
values: [1j, 1j, 1j, 1j, 1j, 1j, 1j, 1j]


In [7]:
import time


def bench(fn, reps: int) -> float:
    '''
    mean wall time of one call of fn in microseconds.
    '''
    for _ in range(2):
        fn()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps * 1e6


many = torch.tensor([[[2, 0], [2, 1]]] * 256, dtype=torch.long, device=CUDA)
batched_us = bench(lambda: fermion.pfaffian(fermion.vacuum_contraction(
    fermion.operator_vectors(coeff, many), 1)), 20)
scalar_us = bench(lambda: [pfaffian(_vacuum_contraction([gamma_0, gamma_1], 1))
                           for _ in range(256)], 2)
print(f'batched, 256 lists : {batched_us:9.1f} us  ({batched_us / 256:.2f} us each)')
print(f'one by one, 256    : {scalar_us:9.1f} us  ({scalar_us / 256:.2f} us each)')
print(f'speedup            : {scalar_us / batched_us:.0f}x')


batched, 256 lists :      22.9 us  (0.09 us each)
one by one, 256    :    7981.3 us  (31.18 us each)
speedup            : 349x
